# TnDrr AI/ML Internship Assignment - Task 1: LoRA Fine-Tuning

**Objective**: Fine-tune `Qwen/Qwen2.5-0.5B-Instruct` on AG News (4-class classification) using Hugging Face Transformers, PEFT, and PyTorch.

### Pipeline Overview:
1. GPU & Environment Setup
2. AG News Dataset Loading & Reproducible Train/Test Split (Seed = 42)
3. Base Model Zero-Shot Evaluation (Held-out test set >= 50 samples)
4. LoRA Adapter Configuration ($r=16, \alpha=32$)
5. LoRA Fine-Tuning with Prompt Loss Masking
6. Fine-Tuned Model Evaluation on Identical Test Set
7. Per-Class Metrics, Confusion Matrix Generation, and Actual Failure Case Analysis
8. Export Results back to local repository

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
!pip install -q torch transformers peft datasets accelerate scikit-learn matplotlib pandas

In [ ]:
# 2. Imports and Reproducibility
import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, PeftModel

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

In [ ]:
# 3. Dataset Configuration & Label Mapping
LABEL_MAPPING = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
LABEL_TO_ID = {v.lower(): k for k, v in LABEL_MAPPING.items()}
LABEL_TO_ID.update({"sci/tech": 3, "scitech": 3, "science/tech": 3, "science": 3, "tech": 3, "technology": 3})

SYSTEM_PROMPT = (
    "You are an expert news classifier. Classify the given news article into exactly one "
    "of the following four categories: World, Sports, Business, Sci/Tech. "
    "Respond with only the category name."
)

def format_instruction_prompt(text: str) -> str:
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\nNews Article: {text.strip()}\nCategory:<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

def parse_predicted_label(prediction_text: str) -> int:
    cleaned = prediction_text.strip().lower()
    for key, label_id in LABEL_TO_ID.items():
        if key in cleaned:
            return label_id
    return -1

# Load AG News
dataset = load_dataset("ag_news")
train_subset = dataset["train"].shuffle(seed=SEED).select(range(2000))
eval_subset = dataset["test"].shuffle(seed=SEED).select(range(100))

eval_records = []
for i, item in enumerate(eval_subset):
    eval_records.append({
        "idx": i,
        "text": item["text"],
        "label_id": item["label"],
        "label_name": LABEL_MAPPING[item["label"]],
        "prompt": format_instruction_prompt(item["text"])
    })

print(f"Loaded {len(train_subset)} training samples and {len(eval_records)} held-out test samples.")

In [ ]:
# 4. Load Base Model and Tokenizer
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)
if not torch.cuda.is_available():
    base_model = base_model.to(device)
print("Base model loaded successfully.")

In [ ]:
# 5. Evaluation Function
def run_eval(model, tag="base_model"):
    model.eval()
    y_true, y_pred, failures = [], [], []
    
    print(f"Evaluating {tag} on {len(eval_records)} samples...")
    with torch.no_grad():
        for i, item in enumerate(eval_records):
            inputs = tokenizer(item["prompt"], return_tensors="pt").to(device)
            out = model.generate(
                **inputs,
                max_new_tokens=8,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            gen_tokens = out[0][inputs["input_ids"].shape[1]:]
            gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
            pred_id = parse_predicted_label(gen_text)
            
            y_true.append(item["label_id"])
            y_pred.append(pred_id)
            
            if pred_id != item["label_id"]:
                failures.append({
                    "idx": item["idx"],
                    "text": item["text"],
                    "true_label": item["label_name"],
                    "pred_label": LABEL_MAPPING.get(pred_id, "Unknown"),
                    "raw_output": gen_text
                })
                
    acc = accuracy_score(y_true, y_pred)
    target_names = [LABEL_MAPPING[k] for k in range(4)]
    report = classification_report(y_true, y_pred, labels=[0,1,2,3], target_names=target_names, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    
    # Plot Confusion Matrix
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.matshow(cm, cmap=plt.cm.Blues)
    for (r, c), val in np.ndenumerate(cm):
        ax.text(c, r, str(val), ha='center', va='center', color='white' if val > cm.max()/2 else 'black')
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(target_names, rotation=30, ha='left')
    ax.set_yticklabels(target_names)
    ax.set_title(f"Confusion Matrix: {tag}")
    plt.tight_layout()
    plt.savefig(f"confusion_matrix_{tag}.png", dpi=150)
    plt.show()
    
    results = {
        "tag": tag,
        "accuracy": float(acc),
        "macro_f1": float(report["macro avg"]["f1-score"]),
        "per_class": {k: report[k] for k in target_names},
        "confusion_matrix": cm.tolist(),
        "total_failures": len(failures),
        "failures": failures
    }
    with open(f"metrics_{tag}.json", "w") as f:
        json.dump(results, f, indent=2)
    print(f"[{tag}] Accuracy: {acc*100:.2f}% | Macro F1: {report['macro avg']['f1-score']:.4f}")
    return results

In [ ]:
# 6. Run Base Model Evaluation
base_results = run_eval(base_model, tag="base_model")

In [ ]:
# 7. Setup LoRA Model
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

In [ ]:
# 8. Dataset Preparation with Target Masking
from torch.utils.data import Dataset as TorchDataset

class AGNewsTrainDataset(TorchDataset):
    def __init__(self, raw_ds, tokenizer, max_length=256):
        self.samples = []
        for row in raw_ds:
            p_str = format_instruction_prompt(row["text"])
            t_str = LABEL_MAPPING[row["label"]] + "<|im_end|>"
            p_ids = tokenizer.encode(p_str, add_special_tokens=False)
            t_ids = tokenizer.encode(t_str, add_special_tokens=False)
            
            ids = (p_ids + t_ids)[:max_length]
            labels = ([-100] * len(p_ids) + t_ids)[:max_length]
            self.samples.append({
                "input_ids": torch.tensor(ids, dtype=torch.long),
                "attention_mask": torch.ones(len(ids), dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)
            })
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

train_ds = AGNewsTrainDataset(train_subset, tokenizer)

In [ ]:
# 9. Train LoRA Adapter
training_args = TrainingArguments(
    output_dir="./lora_checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
)

trainer.train()
peft_model.save_pretrained("./lora_adapter")
tokenizer.save_pretrained("./lora_adapter")
print("LoRA adapter saved.")

In [ ]:
# 10. Run LoRA Model Evaluation
lora_results = run_eval(peft_model, tag="lora_finetuned")

In [ ]:
# 11. Final Metric Comparison
print("=" * 50)
print(f"Base Model Accuracy : {base_results['accuracy']*100:.2f}%")
print(f"LoRA Model Accuracy : {lora_results['accuracy']*100:.2f}%")
print(f"Base Model Macro F1 : {base_results['macro_f1']:.4f}")
print(f"LoRA Model Macro F1 : {lora_results['macro_f1']:.4f}")
print("=" * 50)

# Three actual failure examples
print("\n--- Three Actual Failure Examples (Base or LoRA) ---")
for i, failure in enumerate(lora_results["failures"][:3]):
    print(f"\nFailure #{i+1}:")
    print(f"  Article: {failure['text'][:120]}...")
    print(f"  True Category : {failure['true_label']}")
    print(f"  Predicted     : {failure['pred_label']}")
    print(f"  Raw Output    : '{failure['raw_output']}'")

In [ ]:
# 12. Download Results to Local Machine
from google.colab import files
!zip -r task1_colab_results.zip metrics_base_model.json metrics_lora_finetuned.json confusion_matrix_*.png lora_adapter/
files.download("task1_colab_results.zip")